# Train lev on Gemma 4 E4B in Colab

bf16 LoRA on `google/gemma-4-E4B-it`, the `e4b-instruct` preset ([ADR-031](https://github.com/rcythr/gev/blob/main/docs/DECISIONS.md)).

**Runtime:** Runtime → Change runtime type → **H100**, **A100** or **L4**. A T4 cannot train this: its 16 GB is about E4B's bf16 weights alone, and it has no bf16. `lev plan` estimates, scaled from a measured H100 run and unmeasured on the others: H100 ~9 h, A100 ~28 h, L4 ~72 h. Recheck after step 5 measures your prompt lengths.

**Secrets** (the key icon in the left sidebar), with notebook access on:

- `HF_TOKEN`: a Hugging Face token whose account has accepted the Gemma 4 licence on the model page.
- `GITHUB_TOKEN`: only if `rcythr/gev` is private; a fine-grained token with read access to it.

**Disconnects are expected.** The mixture and checkpoints live on Google Drive. After a disconnect, run every cell again: training resumes from the newest checkpoint, at the step, learning rate and batch order it reached. Each checkpoint is ~1 GB, and `KEEP_LAST` bounds how many stay on Drive.

## 1. Settings and GPU

In [ ]:
import os
import subprocess
import sys

import torch

PRESET = "e4b-instruct"
BRANCH = "main"
REPO = "github.com/rcythr/gev.git"
DRIVE_ROOT = "/content/drive/MyDrive/lev"
RUN_SMOKE = True  # 20 steps on Gemma 4 E2B before the real run: catches setup errors in minutes
KEEP_LAST = 2

assert sys.version_info >= (3, 12), f"lev needs Python 3.12+, this runtime has {sys.version}"
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > H100, A100 or L4"
name = torch.cuda.get_device_name(0)
memory_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
assert torch.cuda.get_device_capability(0)[0] >= 8, (
    f"{name} has no native bf16; use H100, A100 or L4"
)

# Optimiser batch stays 32 rows on every card; smaller cards take it in more micro-batches.
if memory_gb >= 70:
    BATCH, ACCUM = 32, 1
elif memory_gb >= 35:
    BATCH, ACCUM = 16, 2
elif memory_gb >= 20:
    BATCH, ACCUM = 8, 4
else:
    raise SystemExit(f"{name} has {memory_gb:.0f} GB; E4B in bf16 needs at least an L4 (24 GB)")

if "H100" in name:
    GPU = "h100"
elif "A100" in name:
    GPU = "a100-80" if memory_gb >= 70 else "a100-40"
elif "L4" in name:
    GPU = "l4"
else:
    GPU = "a100-40" if memory_gb >= 35 else "l4"  # estimates only

DATA_DRIVE = f"{DRIVE_ROOT}/data/mixture"
DATA = "/content/mixture"  # local copy: reading from Drive mid-run is slow
CKPT = f"{DRIVE_ROOT}/checkpoints/{PRESET}"
SIZING = f"--gpu {GPU} --per-device-batch {BATCH} --grad-accum {ACCUM}"
print(f"{name}, {memory_gb:.0f} GB -> batch {BATCH} x {ACCUM} accumulation, budget as {GPU}")

## 2. Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(CKPT, exist_ok=True)

## 3. Secrets and code

In [ ]:
from google.colab import userdata


def secret(key):
    try:
        return userdata.get(key)
    except Exception:  # missing, or notebook access not granted
        return None


hf_token = secret("HF_TOKEN")
assert hf_token, "Add HF_TOKEN in the Secrets panel and give this notebook access"
os.environ["HF_TOKEN"] = hf_token
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

github_token = secret("GITHUB_TOKEN")
url = f"https://{github_token}@{REPO}" if github_token else f"https://{REPO}"
if not os.path.isdir("/content/gev"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/content/gev"], check=True)
else:
    subprocess.run(["git", "-C", "/content/gev", "pull", url, BRANCH], check=True)
# Keep the token out of the clone's config.
subprocess.run(
    ["git", "-C", "/content/gev", "remote", "set-url", "origin", f"https://{REPO}"], check=True
)
!git -C /content/gev log --oneline -1

## 4. Install

Versions match `uv.lock`. Colab's own torch is kept. pip may warn about conflicts with preinstalled packages this notebook does not use.

In [ ]:
!pip install -q -e "/content/gev/packages/lev[train]" \
    "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0" \
    "datasets==5.0.1" "huggingface-hub==1.32.0" "safetensors==0.8.0"
!lev --help | head -3
# Fails here, not hours in, if the token's account has not accepted the Gemma licence.
!python -c "import huggingface_hub as h; print(h.model_info('google/gemma-4-E4B-it').id, 'ok')"

## 5. Training data

Built once, from 29 public sources, then kept on Drive. The first build takes a while.

In [ ]:
if not os.path.isfile(f"{DATA_DRIVE}/train.jsonl"):
    !lev data build --out {DATA}
    !mkdir -p {DATA_DRIVE} && cp -r {DATA}/. {DATA_DRIVE}/
elif not os.path.isfile(f"{DATA}/train.jsonl"):
    !mkdir -p {DATA} && cp -r {DATA_DRIVE}/. {DATA}/
!ls -la {DATA}

In [ ]:
# Measures real prompt lengths under Gemma's tokenizer, so the time estimate is this run's.
!lev plan --preset {PRESET} --data {DATA} {SIZING}

## 6. Smoke run

20 steps on Gemma 4 E2B. It fails within minutes if anything is wrong, including the prompt framing: if the `gemma` framing differs from the tokenizer's chat template, training stops and prints both. Once it has passed, set `RUN_SMOKE = False` so a resumed session goes straight to training.

In [ ]:
if RUN_SMOKE:
    !lev train --preset smoke --data {DATA} --output-dir /content/lev-smoke --max-steps 20 --fresh

## 7. Train

Resumes from the newest checkpoint in `CKPT` on Drive. Keep this tab open; Colab disconnects idle sessions.

In [ ]:
!lev train --preset {PRESET} --data {DATA} --output-dir {CKPT} {SIZING} --keep-last {KEEP_LAST}

## 8. Calibrate

Fits one temperature per question bucket on the calibration split and writes `calibration.json` beside the checkpoints.

In [ ]:
!lev calibrate --preset {PRESET} --data {DATA} --output-dir {CKPT}

## 9. Try it

In [ ]:
sys.path.insert(0, "/content/gev/packages/lev/src")
import lev

model = lev.load(CKPT, model_id="google/gemma-4-E4B-it", prompt_style="gemma")
result = model.system_one(
    "Hi, I was charged twice for my order #4471 and I want a refund.",
    {
        "intent": {
            "type": "choice",
            "instructions": "What does the customer want?",
            "criteria": {
                "refund": "wants money back",
                "cancel": None,
                "track": None,
                "other": None,
            },
        },
        "urgent": {"type": "noul", "instructions": "Does this need a human within the hour?"},
    },
)
print(result.answers["intent"].choice, result.answers["intent"].probabilities)
print("urgent:", result.answers["urgent"].noul)

## 10. Package a release

One flat directory on Drive with the adapter, Mode B head, tokenizer, calibration and a manifest. `lev serve --checkpoint <dir>` serves it. To publish: `lev release publish <dir> --repo <org/name>`.

In [ ]:
del model
torch.cuda.empty_cache()
!lev release build --checkpoint {CKPT} --out {DRIVE_ROOT}/releases/{PRESET} --preset {PRESET}